In [0]:
ors_api_key = dbutils.secrets.get(
    scope="shipp",
    key="ors_api_key"
)

In [0]:
%sql
select * from bootcamp_students.shipp_silver.silver_listings;



In [0]:
%sql
select * from bootcamp_students.shipp_silver.silver_requests;

In [0]:
from pyspark.sql import functions as F

LISTINGS = "bootcamp_students.shipp_silver.silver_listings"
REQUESTS = "bootcamp_students.shipp_silver.silver_requests"

listings = spark.table(LISTINGS).alias("l")
requests = spark.table(REQUESTS).alias("r")

eligible_pairs = (
    requests
    .join(
        listings,
        (F.upper(F.col("r.category")) == F.upper(F.col("l.category")))
        & (F.upper(F.col("r.status")) == "OPEN")
        & (F.upper(F.col("l.status")) == "AVAILABLE")
        & F.col("r.latitude").isNotNull()
        & F.col("r.longitude").isNotNull()
        & F.col("l.latitude").isNotNull()
        & F.col("l.longitude").isNotNull(),
        "inner"
    )
    .select(
        F.col("r.request_id"),
        F.col("l.listing_id"),

        F.col("r.latitude").alias("request_latitude"),
        F.col("r.longitude").alias("request_longitude"),

        F.col("l.latitude").alias("listing_latitude"),
        F.col("l.longitude").alias("listing_longitude"),

        F.col("r.category").alias("request_category"),
        F.col("l.category").alias("listing_category"),

        F.col("r.status").alias("request_status"),
        F.col("l.status").alias("listing_status")
    )
)

print("Eligible ORS pair count:", eligible_pairs.count())

display(
    eligible_pairs.orderBy("request_id", "listing_id")
)

In [0]:
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

w.secrets.put_secret("shipp","ors-api-key",string_value ="eyJvcmciOiI1YjNjZTM1OTc4NTExMTAwMDFjZjYyNDgiLCJpZCI6IjgyNDA4ZmZjZjI4ZTRlNjI4MTZlMTI0M2Y0Y2I1Mzk4IiwiaCI6Im11cm11cjY0In0=")


In [0]:
ors_api_key = dbutils.secrets.get(
    scope="shipp",
    key="ors-api-key"
)

secret_is_valid = bool(ors_api_key and ors_api_key.strip())

print("ORS secret retrieved:", secret_is_valid)

In [0]:
import requests
import time

ORS_MATRIX_URL = "https://api.openrouteservice.org/v2/matrix/driving-car"

# IMPORTANT:
# Use the same secure ORS key retrieval that already worked
# in your successful 20_ors_smoke_test.py.
#
# Example ONLY if your Databricks secret is already configured:
ORS_API_KEY = dbutils.secrets.get(scope="shipp", key="ors_api_key")

if "ORS_API_KEY" not in globals():
    raise RuntimeError(
        "ORS_API_KEY is not loaded. Use the same secure key retrieval "
        "mechanism that passed your ORS smoke test."
    )


def call_ors_matrix(
    listing_lon,
    listing_lat,
    request_lon,
    request_lat,
    max_attempts=3,
):
    """
    Listing location = origin.
    Request location = destination.

    Returns the raw ORS JSON response.
    """

    payload = {
        "locations": [
            [float(listing_lon), float(listing_lat)],
            [float(request_lon), float(request_lat)],
        ],
        "sources": [0],
        "destinations": [1],
        "metrics": ["distance", "duration"],
        "units": "m",
    }

    headers = {
        "Authorization": ORS_API_KEY,
        "Content-Type": "application/json",
    }

    last_error = None

    for attempt in range(1, max_attempts + 1):
        try:
            response = requests.post(
                ORS_MATRIX_URL,
                json=payload,
                headers=headers,
                timeout=30,
            )

            response.raise_for_status()

            raw_response = response.json()

            distance_m = raw_response["distances"][0][0]
            duration_sec = raw_response["durations"][0][0]

            if distance_m is None or duration_sec is None:
                raise ValueError(
                    "ORS returned null distance or duration."
                )

            return {
                "http_status": response.status_code,
                "request_payload": payload,
                "raw_response": raw_response,
                "distance_m": float(distance_m),
                "duration_sec": float(duration_sec),
            }

        except Exception as exc:
            last_error = exc

            if attempt < max_attempts:
                time.sleep(2 ** (attempt - 1))

    raise RuntimeError(
        f"ORS failed after {max_attempts} attempts: {last_error}"
    )